# Module 5 | Class 5 Assignment: Anomaly Detection + Association Rules
**Related slides:** Class 5 — Anomaly Detection + Association Rules  
**Estimated time:** 90 minutes (two activities)

### Objective
Use Isolation Forest to detect fraud in credit card data (Activity 3), and compute association rules by hand and with code (Activity 4).

---

# Activity 3: Fraud Detection with Isolation Forest

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## Task 1: Load and Explore

**What to do:** Load the Credit Card Fraud dataset and understand the class imbalance.

- Step 1: Load CSV into a DataFrame. Check shape (~285,000 rows).
- Step 2: Print `df['Class'].value_counts()` to see the fraud rate.
- Step 3: Note: V1–V28 are PCA-transformed features (already scaled). Amount and Time are not.

In [ ]:
# Load the dataset (upload creditcard.csv to Colab first)
df = pd.read_csv('creditcard.csv')

# Step 1: Check shape
print('Shape:', df.shape)
print()

# Step 2: Class distribution
print('Class distribution:')
print(df['Class'].value_counts())
print()
print('Fraud rate: {:.4f}%'.format(df['Class'].mean() * 100))

# Step 3: Preview
df.head()

## Task 2: Prepare the Data

**What to do:** Scale Amount and Time, then split the data.

- Step 1: Apply `StandardScaler()` to Amount and Time columns.
- Step 2: Split 80/20 with `stratify=y` to preserve the fraud ratio.

In [ ]:
# Step 1: Scale Amount and Time
scaler = StandardScaler()
df['scaled_Amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_Time']   = scaler.fit_transform(df[['Time']])
df.drop(columns=['Amount', 'Time'], inplace=True)

# Step 2: Define features and target
X = df.drop(columns=['Class'])
y = df['Class']

# Train/test split — stratified to preserve fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('Fraud in test :', y_test.sum())

## Task 3: Apply Isolation Forest

**What to do:** Fit Isolation Forest with different contamination rates and compare.

- Step 1: Fit with `contamination=0.00173` (actual fraud rate). Predict on test set. Map -1 to fraud.
- Step 2: Repeat with `contamination=0.01` and `contamination=0.005`.
- Step 3: Compare precision, recall, F1 across all three settings.

In [ ]:
contamination_rates = {
    'contamination=0.00173 (actual fraud rate)': 0.00173,
    'contamination=0.005':                        0.005,
    'contamination=0.01':                         0.01,
}

results = {}

for label, cont in contamination_rates.items():
    model = IsolationForest(n_estimators=100, contamination=cont, random_state=42, n_jobs=-1)
    model.fit(X_train)
    raw_preds = model.predict(X_test)
    # Map: -1 (anomaly) -> 1 (fraud), 1 (normal) -> 0
    preds = np.where(raw_preds == -1, 1, 0)
    results[label] = {'model': model, 'preds': preds}
    print(f'\n=== {label} ===')
    print(classification_report(y_test, preds, target_names=['Normal', 'Fraud']))

## Task 4: Evaluate

**What to do:** Compute precision, recall, F1, and plot the confusion matrix.

- Step 1: Use `classification_report(y_test, predictions)`.
- Step 2: Plot confusion matrix with `ConfusionMatrixDisplay.from_predictions()`.
- Step 3: Discuss the precision-recall tradeoff as contamination changes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (label, res) in zip(axes, results.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, res['preds'],
        display_labels=['Normal', 'Fraud'],
        colorbar=False, ax=ax
    )
    ax.set_title(label, fontsize=8)

plt.suptitle('Confusion Matrices — Isolation Forest (3 contamination rates)', fontsize=11)
plt.tight_layout()
plt.show()

### Precision-Recall Tradeoff Discussion

- **Low contamination (0.00173):** The model flags very few transactions as fraud, so precision is higher but recall is lower — many actual frauds are missed.
- **Higher contamination (0.01):** More transactions are flagged, increasing recall (catching more real fraud) but decreasing precision (more false alarms).
- **Key insight:** In fraud detection, **recall** is usually more important — missing fraud is costlier than a false alarm. The right contamination depends on business tolerance for false positives vs. missed fraud.

## Task 5: Rank by Anomaly Score

**What to do:** Extract anomaly scores and evaluate precision-at-k.

- Step 1: Get scores with `model.decision_function(X_test)` (lower = more anomalous).
- Step 2: Sort by score. Of the top 100 most anomalous transactions, how many are actually fraud?
- Step 3: Plot precision-at-k for k = 10, 50, 100, 200, 500.

In [ ]:
# Use the model with actual fraud rate contamination
best_model = results['contamination=0.00173 (actual fraud rate)']['model']

# Step 1: Get anomaly scores (lower = more anomalous)
scores = best_model.decision_function(X_test)

# Step 2: Sort ascending (most anomalous first)
sorted_indices = np.argsort(scores)
y_test_array = np.array(y_test)

top_100_fraud = y_test_array[sorted_indices[:100]].sum()
print(f'Top 100 most anomalous transactions: {top_100_fraud} are actual fraud')
print(f'Precision@100 = {top_100_fraud / 100:.2f}')

# Step 3: Precision-at-k plot
k_values = [10, 50, 100, 200, 500]
precision_at_k = []

for k in k_values:
    top_k_fraud = y_test_array[sorted_indices[:k]].sum()
    precision_at_k.append(top_k_fraud / k)
    print(f'Precision@{k:>4} = {top_k_fraud}/{k} = {top_k_fraud/k:.2f}')

plt.figure(figsize=(7, 4))
plt.plot(k_values, precision_at_k, marker='o', color='steelblue', linewidth=2)
plt.xlabel('k (top-k most anomalous)')
plt.ylabel('Precision@k')
plt.title('Precision-at-k — Isolation Forest Anomaly Scores')
plt.xticks(k_values)
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---

# Activity 4: Association Rule Mining

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

## Task 1: Manual Computation

**Transaction table:**

| Transaction | Items |
|-------------|-------|
| T1 | Bread, Butter, Milk |
| T2 | Bread, Butter |
| T3 | Bread, Milk, Tea |
| T4 | Butter, Milk, Tea |
| T5 | Bread, Butter, Milk, Tea |
| T6 | Bread, Tea |
| T7 | Milk, Tea |
| T8 | Bread, Butter, Milk |
| T9 | Bread, Milk |
| T10 | Butter, Tea |

---

### Step 1 — Support for each item (Count / 10)

| Item | Transactions | Support |
|------|-------------|--------|
| Bread | T1,T2,T3,T5,T6,T8,T9 | 7/10 = **0.70** |
| Butter | T1,T2,T4,T5,T8,T10 | 6/10 = **0.60** |
| Milk | T1,T3,T4,T5,T7,T8,T9 | 7/10 = **0.70** |
| Tea | T3,T4,T5,T6,T7,T10 | 6/10 = **0.60** |

---

### Step 2 — Support for pairs

| Pair | Transactions | Support |
|------|-------------|--------|
| {Bread, Butter} | T1,T2,T5,T8 | 4/10 = **0.40** |
| {Bread, Milk} | T1,T3,T5,T8,T9 | 5/10 = **0.50** |
| {Milk, Tea} | T3,T4,T5,T7 | 4/10 = **0.40** |

---

### Step 3 — Rule: Bread => Butter

- **Confidence** = support({Bread,Butter}) / support(Bread) = 0.40 / 0.70 = **0.571**
- **Lift** = confidence / support(Butter) = 0.571 / 0.60 = **0.952**

### Step 4 — Rule: Butter => Bread

- **Confidence** = support({Bread,Butter}) / support(Butter) = 0.40 / 0.60 = **0.667**
- **Lift** = confidence / support(Bread) = 0.667 / 0.70 = **0.952**

> **Note:** Lift is the same in both directions (it is symmetric), but confidence differs because the antecedent support differs. Confidence is directional.

## Task 2: Identify Strong Rules

**Criteria:** min support = 30%, min confidence = 50%

**Frequent itemsets (support ≥ 0.30):**

| Itemset | Support |
|---------|--------|
| {Bread} | 0.70 |
| {Butter} | 0.60 |
| {Milk} | 0.70 |
| {Tea} | 0.60 |
| {Bread, Butter} | 0.40 |
| {Bread, Milk} | 0.50 |
| {Milk, Tea} | 0.40 |

**Rules that survive (confidence ≥ 0.50):**

| Rule | Support | Confidence | Lift |
|------|---------|-----------|------|
| Butter => Bread | 0.40 | 0.667 | 0.952 |
| Bread => Milk | 0.50 | 0.714 | 1.020 |
| Milk => Bread | 0.50 | 0.714 | 1.020 |
| Milk => Tea | 0.40 | 0.571 | 0.952 |
| Tea => Milk | 0.40 | 0.667 | 0.952 |

> **Highest lift:** {Bread, Milk} rules (lift ≈ 1.02) — Bread and Milk have a slight positive association. All other surviving rules have lift < 1 (slight negative association despite meeting confidence threshold).

## Task 3: Code Verification

**What to do:** Verify manual calculations using mlxtend.

- Step 1: Encode as one-hot DataFrame using `TransactionEncoder`.
- Step 2: Run `apriori(df, min_support=0.3, use_colnames=True)`.
- Step 3: Run `association_rules(frequent_items, metric='confidence', min_threshold=0.5)`.
- Step 4: Compare to manual calculations.

In [ ]:
# Step 1: Define transactions
transactions = [
    ['Bread', 'Butter', 'Milk'],
    ['Bread', 'Butter'],
    ['Bread', 'Milk', 'Tea'],
    ['Butter', 'Milk', 'Tea'],
    ['Bread', 'Butter', 'Milk', 'Tea'],
    ['Bread', 'Tea'],
    ['Milk', 'Tea'],
    ['Bread', 'Butter', 'Milk'],
    ['Bread', 'Milk'],
    ['Butter', 'Tea'],
]

# One-hot encode
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_enc = pd.DataFrame(te_array, columns=te.columns_)
print('Encoded transaction table:')
print(df_enc.to_string())

In [ ]:
# Step 2: Frequent itemsets
frequent_items = apriori(df_enc, min_support=0.3, use_colnames=True)
frequent_items = frequent_items.sort_values('support', ascending=False).reset_index(drop=True)
print('Frequent itemsets (min support = 0.3):')
print(frequent_items.to_string())

In [ ]:
# Step 3: Association rules
rules = association_rules(frequent_items, metric='confidence', min_threshold=0.5)
rules = rules[['antecedents','consequents','support','confidence','lift']].sort_values('lift', ascending=False)
print('Association rules (min confidence = 0.5):')
print(rules.to_string(index=False))

### Step 4 — Comparison: Manual vs Code

| Rule | Manual Confidence | Code Confidence | Manual Lift | Code Lift |
|------|------------------|----------------|------------|----------|
| Bread => Butter | 0.571 | 0.571 | 0.952 | 0.952 |
| Butter => Bread | 0.667 | 0.667 | 0.952 | 0.952 |
| Bread => Milk | 0.714 | 0.714 | 1.020 | 1.020 |
| Milk => Bread | 0.714 | 0.714 | 1.020 | 1.020 |
| Milk => Tea | 0.571 | 0.571 | 0.952 | 0.952 |
| Tea => Milk | 0.667 | 0.667 | 0.952 | 0.952 |

> ✅ Manual calculations match the mlxtend code output exactly.

## Task 4: Telecom Analogy

**What to do:** Map items to Uztelecom services and write 2-3 business recommendations based on the strongest rules.

### Service Mapping

| Grocery Item | Uztelecom Service |
|-------------|------------------|
| Bread | Internet (Home Broadband) |
| Milk | Mobile (SIM / Mobile Data) |
| Butter | IPTV |
| Tea | Cloud Storage / Extra Backup |

### Business Recommendations

1. **Bundle Internet + Mobile** *(Bread => Milk, lift ≈ 1.02)*: Customers who subscribe to home broadband very frequently also take a mobile data plan. Uztelecom should offer a discounted Internet + Mobile bundle to increase uptake of both services simultaneously.

2. **Upsell IPTV to Internet customers** *(Butter => Bread, confidence = 0.667)*: Most IPTV subscribers already have broadband. Target IPTV-only or new IPTV customers with promotional broadband upgrade offers to consolidate their services under one account.

3. **Promote Cloud Storage alongside Mobile** *(Milk => Tea / Tea => Milk)*: Mobile users and cloud storage subscribers frequently overlap. Offering free trial cloud backup to mobile customers (and vice versa) can raise conversion rates and reduce churn by increasing service dependency.